# End-to-End High-Dimensional Medical Data Pipeline: From-Scratch PCA & Gaussian Naive Bayes Optimization

### 1. Load Complex Medical Dataset

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import precision_score, recall_score, f1_score

# 1. Load Real Complex Medical Dataset: UCI Arrhythmia (279 attributes)
data_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/arrhythmia/arrhythmia.data"
raw_df = pd.read_csv(data_url, header=None, na_values='?')

X_raw = raw_df.iloc[:, :-1].values.astype(float)
# Binary classification: 1 = Normal ECG, >1 = Arrhythmia / Cardiac Abnormality
y_raw = (raw_df.iloc[:, -1].values > 1).astype(int)

print(f"Dataset Extracted: {X_raw.shape[0]} patient records with {X_raw.shape[1]} clinical & ECG features.")
print(f"Real Missing Entries Detected: {np.isnan(X_raw).sum()}")

Dataset Extracted: 452 patient records with 279 clinical & ECG features.
Real Missing Entries Detected: 408


### 2. Statistical Imputation & Feature Filtering (From Scratch)

In [2]:
# 2. Statistical Imputation - Median strategy to resist heavy outliers in ECG readings
def statistical_imputation(X):
    X_imputed = X.copy()
    for j in range(X.shape[1]):
        col = X_imputed[:, j]
        if np.isnan(col).any():
            median_val = np.nanmedian(col)
            X_imputed[np.isnan(col), j] = np.nan_to_num(median_val, nan=0.0)
    return X_imputed

X_imputed = statistical_imputation(X_raw)
print(f"Missing Entries after Statistical Imputation: {np.isnan(X_imputed).sum()}")

# 3. Remove zero-variance channels (leads with constant signal)
variances = np.var(X_imputed, axis=0)
non_zero_mask = variances > 1e-6
X_filtered = X_imputed[:, non_zero_mask]

print(f"Removed {np.sum(~non_zero_mask)} zero-variance degenerate columns.")
print(f"Cleaned High-Dimensional Matrix Shape: {X_filtered.shape}")

Missing Entries after Statistical Imputation: 0
Removed 17 zero-variance degenerate columns.
Cleaned High-Dimensional Matrix Shape: (452, 262)


### 3. Z-Score Standardization (From Scratch)

In [3]:
# 4. Z-Score Standardization - Implemented from the ground up
def z_score_standardize(X):
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)
    sigma[sigma == 0] = 1.0  # Guard against division by zero
    return (X - mu) / sigma

X_scaled = z_score_standardize(X_filtered)
print(f"Scaled Feature Matrix: Mean = {np.mean(X_scaled):.4f}, Std = {np.mean(np.std(X_scaled, axis=0)):.4f}")

Scaled Feature Matrix: Mean = 0.0000, Std = 1.0000


### 4. PCA & Whitening via Eigenvalue Decomposition (From Scratch)

In [4]:
# 5. Custom PCA and Whitening Implementation
def compute_pca_whitening(X, n_components=30):
    n_samples = X.shape[0]
    
    # Step A: Sample Covariance Matrix (X is already mean-centered)
    cov_mat = np.dot(X.T, X) / (n_samples - 1)
    
    # Step B: Eigenvalue Decomposition
    eigenvalues, eigenvectors = np.linalg.eigh(cov_mat)
    
    # Step C: Sort descending to get principal components
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]
    
    # Step D: Extract top-k subspace
    top_evals = np.maximum(eigenvalues[:n_components], 1e-9)
    top_evecs = eigenvectors[:, :n_components]
    
    # Step E: Projection & Whitening transformation (normalizing variance)
    X_projected = np.dot(X, top_evecs)
    X_whitened = X_projected / np.sqrt(top_evals)
    
    var_explained = np.sum(top_evals) / np.sum(eigenvalues)
    return X_whitened, var_explained, top_evals

X_whitened, total_var, top_evals = compute_pca_whitening(X_scaled, n_components=30)
print(f"Covariance Matrix Shape: ({X_scaled.shape[1]}, {X_scaled.shape[1]})")
print(f"Total Retained Variance across 30 Components: {total_var * 100:.2f}%")
print(f"Whitened Subspace Matrix Shape: {X_whitened.shape}")
print(f"Component Variances (Unit Sphered): {np.round(np.var(X_whitened, axis=0)[:5], 4)} ...")

Covariance Matrix Shape: (262, 262)
Total Retained Variance across 30 Components: 66.50%
Whitened Subspace Matrix Shape: (452, 30)
Component Variances (Unit Sphered): [0.9978 0.9978 0.9978 0.9978 0.9978] ...


### 5. Supervised Learning & Optimization (Gaussian Naive Bayes)

In [5]:
# 6. Stratified Train/Test Partition
X_train, X_test, y_train, y_test = train_test_split(
    X_whitened, y_raw, test_size=0.20, random_state=42, stratify=y_raw
)

# 7. Grid Search with 5-Fold Stratified CV to combat high variance and mitigate overfitting
# For GaussianNB, we tune the 'var_smoothing' parameter
param_grid = {
    'var_smoothing': [1e-9, 1e-7, 1e-5, 1e-3, 1e-1, 1.0, 10.0]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
clf = GaussianNB()

grid_search = GridSearchCV(clf, param_grid, cv=cv, scoring='f1', n_jobs=-1)
grid_search.fit(X_train, y_train)

best_clf = grid_search.best_estimator_
print(f"Optimal Hyperparameters (Grid Search): {grid_search.best_params_}")
print(f"Cross-Validated F1-Score: {grid_search.best_score_:.4f}")

Optimal Hyperparameters (Grid Search): {'var_smoothing': 0.1}
Cross-Validated F1-Score: 0.6952


### 6. Systematic Metric Evaluation

In [6]:
# 8. Rigorous Metric Evaluation on Unseen Test Partition
y_pred = best_clf.predict(X_test)

print("-" * 50)
print("EVALUATION ON UNSEEN TEST HOLDOUT (452-patient Arrhythmia)")
print("-" * 50)
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred):.4f}")

--------------------------------------------------
EVALUATION ON UNSEEN TEST HOLDOUT (452-patient Arrhythmia)
--------------------------------------------------
Precision: 0.7027
Recall:    0.6190
F1-Score:  0.6582
